In [0]:
WITH base AS (
    SELECT
        t.CDE_ID,
        d.datefull,
        t.LocationId,
        l.LC_Trade_Brand_Description,
        SUM(t.Net_Revenue) AS Revenue,
        CASE
            WHEN d.datefull BETWEEN '2025-01-01' AND '2025-03-31' THEN 'Q1_2025'
            WHEN d.datefull BETWEEN '2026-01-01' AND '2026-03-31' THEN 'Q1_2026'
            ELSE 'None'
        END AS Offer_Flag
    FROM       cpx_dataanalytics_gold.customer360.c360_f_transaction     t
    LEFT JOIN  cpx_dataanalytics_gold.customer360.cpx_d_location l ON t.LocationId  = l.LocationID
    LEFT JOIN  cpx_dataanalytics_gold.customer360.cpx_d_date      d ON t.VisitDateId = d.dateid
    WHERE (
            d.datefull BETWEEN '2025-01-01' AND '2025-03-31'
         OR d.datefull BETWEEN '2026-01-01' AND '2026-03-31'
    )
    AND l.LC_Location_Type_Description = 'Restaurant/Store/Games'
    AND t.SCENEFLAG = 'Yes'
    GROUP BY
        t.CDE_ID,
        d.datefull,
        t.LocationId,
        l.LC_Trade_Brand_Description,
        Offer_Flag
),

visit_counts AS (
    SELECT
        CDE_ID,
        Offer_Flag,
        LC_Trade_Brand_Description,
        COUNT(DISTINCT datefull) AS Total_Visits,
        SUM(Revenue)             AS Revenue
    FROM  base
    WHERE Offer_Flag != 'None'
    GROUP BY CDE_ID, Offer_Flag, LC_Trade_Brand_Description
)

SELECT
    Offer_Flag,
    LC_Trade_Brand_Description,
    COUNT(DISTINCT CDE_ID)  AS Unique_Members,
    SUM(Total_Visits)       AS Total_Visits,
    ROUND(SUM(Revenue), 2)  AS Total_Revenue
FROM visit_counts
GROUP BY Offer_Flag, LC_Trade_Brand_Description
ORDER BY Offer_Flag, LC_Trade_Brand_Description;

WITH base AS (
    SELECT
        t.CDE_ID,
        d.datefull,
        t.LocationId,
        l.LC_Trade_Brand_Description,
        SUM(t.Net_Revenue) AS Revenue,
        CASE
            WHEN d.datefull BETWEEN '2025-01-01' AND '2025-03-31' THEN 'Q1_2025'
            WHEN d.datefull BETWEEN '2026-01-01' AND '2026-03-31' THEN 'Q1_2026'
            ELSE 'None'
        END AS Offer_Flag
    FROM       cpx_dataanalytics_gold.customer360.c360_f_transaction     t
    LEFT JOIN  cpx_dataanalytics_gold.customer360.cpx_d_location l ON t.LocationId  = l.LocationID
    LEFT JOIN  cpx_dataanalytics_gold.customer360.cpx_d_date     d ON t.VisitDateId = d.dateid
    WHERE (
            d.datefull BETWEEN '2025-01-01' AND '2025-03-31'
         OR d.datefull BETWEEN '2026-01-01' AND '2026-03-31'
    )
    AND l.LC_Location_Type_Description = 'Restaurant/Store/Games'
    AND t.SCENEFLAG = 'Yes'
    GROUP BY
        t.CDE_ID,
        d.datefull,
        t.LocationId,
        l.LC_Trade_Brand_Description,
        Offer_Flag
),

visit_counts AS (
    SELECT
        CDE_ID,
        Offer_Flag,
        LC_Trade_Brand_Description,
        COUNT(DISTINCT datefull) AS Total_Visits
    FROM  base
    WHERE Offer_Flag != 'None'
    GROUP BY CDE_ID, Offer_Flag, LC_Trade_Brand_Description
)

SELECT
    Offer_Flag,
    LC_Trade_Brand_Description,
    CASE WHEN Total_Visits > 1 THEN 'Repeat Visitor' ELSE '1-Time Visitor' END AS Segment,
    COUNT(DISTINCT CDE_ID) AS Member_Count
FROM visit_counts
GROUP BY
    Offer_Flag,
    LC_Trade_Brand_Description,
    CASE WHEN Total_Visits > 1 THEN 'Repeat Visitor' ELSE '1-Time Visitor' END
ORDER BY Offer_Flag, LC_Trade_Brand_Description, Segment;